In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DATA_DIR = Path("data")

def load_data():
    data_paths = {
        'combination_response': DATA_DIR / 'processed_combination_response_r070_clean_with_qc.csv',
        'drug_features': DATA_DIR / 'Graphlet_features_6_standardized.csv',
        'cell_features': DATA_DIR / 'cell_features_977d.csv'
    }

    print(f"Notebook working directory: {Path.cwd()}")
    print(f"Data directory: {DATA_DIR}")

    missing_files = [path for path in data_paths.values() if not path.is_file()]
    if missing_files:
        missing_text = '\n'.join(f'  - {path}' for path in missing_files)
        raise FileNotFoundError(
            'Required data files were not found.\n'
            f'Please check the DATA_DIR setting: {DATA_DIR}\n'
            f'Missing files:\n{missing_text}'
        )

    combo_df = pd.read_csv(data_paths['combination_response'], encoding='utf-8-sig')

    combo_df = combo_df.rename(columns={
        'target': 'X/X0',
        'drugA_conc': 'drugA Conc (µM)',
        'drugB_conc': 'drugB Conc (µM)'
    })

    combo_df['drugA_name'] = combo_df['drugA_name'].astype(str).str.strip().str.upper()
    combo_df['drugB_name'] = combo_df['drugB_name'].astype(str).str.strip().str.upper()
    combo_df['cell_line'] = combo_df['cell_line'].astype(str).str.strip()

    drug_features = pd.read_csv(data_paths['drug_features'], encoding='utf-8-sig')
    drug_features['name'] = drug_features['name'].astype(str).str.strip().str.upper()
    drug_features.set_index('name', inplace=True)

    cell_features = pd.read_csv(data_paths['cell_features'], encoding='utf-8-sig')
    cell_features['Cell_Line'] = cell_features['Cell_Line'].astype(str).str.strip()
    cell_features.set_index('Cell_Line', inplace=True)

    return combo_df, drug_features, cell_features

def preprocess_data(combo_df, drug_features, cell_features):
    valid_mask = (
        combo_df['drugA_name'].isin(drug_features.index) &
        combo_df['drugB_name'].isin(drug_features.index) &
        combo_df['cell_line'].isin(cell_features.index)
    )

    skipped_count = len(combo_df) - valid_mask.sum()
    print(f"Skipped {skipped_count} samples due to missing features")

    data_df = combo_df.loc[valid_mask, [
        'drugA_name', 'drugB_name', 'cell_line',
        'drugA Conc (µM)', 'drugB Conc (µM)',
        'X/X0', 'single_resp_1', 'single_resp_2'
    ]].copy()

    data_df = data_df.rename(columns={
        'drugA Conc (µM)': 'drugA_conc',
        'drugB Conc (µM)': 'drugB_conc',
        'X/X0': 'target'
    })

    data_df = data_df.reset_index(drop=True)
    data_df['row_id'] = np.arange(len(data_df), dtype=np.int64)
    return data_df

def cell_line_train_val_test_split(
    data_df, train_size=0.70, val_size=0.10, test_size=0.20, random_state=SEED
):

    if not np.isclose(train_size + val_size + test_size, 1.0):
        raise ValueError('train_size + val_size + test_size must equal 1.0')
    if min(train_size, val_size, test_size) <= 0:
        raise ValueError('train_size, val_size and test_size must all be positive')

    grouped_df = data_df.copy()
    outer_split = GroupShuffleSplit(
        n_splits=1, test_size=test_size, random_state=random_state
    )
    train_val_idx, test_idx = next(
        outer_split.split(grouped_df, groups=grouped_df['cell_line'])
    )
    train_val_df = grouped_df.iloc[train_val_idx].copy()
    test_df = grouped_df.iloc[test_idx].copy().reset_index(drop=True)

    relative_val_size = val_size / (train_size + val_size)
    inner_split = GroupShuffleSplit(
        n_splits=1, test_size=relative_val_size, random_state=random_state + 1
    )
    train_idx, val_idx = next(
        inner_split.split(train_val_df, groups=train_val_df['cell_line'])
    )
    train_df = train_val_df.iloc[train_idx].copy().reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].copy().reset_index(drop=True)

    train_cells = set(train_df['cell_line'])
    val_cells = set(val_df['cell_line'])
    test_cells = set(test_df['cell_line'])
    train_rows = set(train_df['row_id'])
    val_rows = set(val_df['row_id'])
    test_rows = set(test_df['row_id'])
    all_rows = set(grouped_df['row_id'])

    audit = pd.DataFrame([
        {'check': 'Train intersection Validation cell lines', 'unexpected_count': len(train_cells & val_cells)},
        {'check': 'Train intersection Test cell lines', 'unexpected_count': len(train_cells & test_cells)},
        {'check': 'Validation intersection Test cell lines', 'unexpected_count': len(val_cells & test_cells)},
        {'check': 'Train intersection Validation rows', 'unexpected_count': len(train_rows & val_rows)},
        {'check': 'Train intersection Test rows', 'unexpected_count': len(train_rows & test_rows)},
        {'check': 'Validation intersection Test rows', 'unexpected_count': len(val_rows & test_rows)},
        {'check': 'Unassigned rows', 'unexpected_count': len(all_rows - (train_rows | val_rows | test_rows))},
        {'check': 'Rows outside original data', 'unexpected_count': len((train_rows | val_rows | test_rows) - all_rows)}
    ])
    audit['passed'] = audit['unexpected_count'].eq(0)
    assert audit['passed'].all(), f'Cell-line split audit failed:\n{audit}'
    assert train_cells | val_cells | test_cells == set(grouped_df['cell_line'])

    split_summary = pd.DataFrame({
        'split': ['Train', 'Validation', 'Test'],
        'samples': [len(train_df), len(val_df), len(test_df)],
        'cell_lines': [len(train_cells), len(val_cells), len(test_cells)]
    })
    split_summary['sample_percent'] = (
        split_summary['samples'] / len(grouped_df) * 100
    ).round(4)
    split_summary['cell_line_percent'] = (
        split_summary['cell_lines'] / grouped_df['cell_line'].nunique() * 100
    ).round(4)

    split_summary.to_csv(
        f'cell_line_cold_start_split_summary_seed{SEED}.csv', index=False, encoding='utf-8-sig'
    )
    audit.to_csv(
        f'cell_line_cold_start_split_audit_seed{SEED}.csv', index=False, encoding='utf-8-sig'
    )
    print('\nCell-line cold-start split summary:')
    print(split_summary.to_string(index=False))
    print('\nAUDIT PASSED: no cell line or sample row is shared across subsets.')
    print(audit.to_string(index=False))
    return train_df, val_df, test_df, split_summary

class DrugCombinationDataset(Dataset):
    def __init__(self, data, drug_features, cell_features):
        self.data = data.reset_index(drop=True)

        self.drug_features = {
            name: row.values[2:].astype(np.float32)
            for name, row in drug_features.iterrows()
        }

        self.cell_features = {
            name: row.values.astype(np.float32)
            for name, row in cell_features.iterrows()
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data.iloc[idx]

        drugA_feat = self.drug_features[sample['drugA_name']]
        drugB_feat = self.drug_features[sample['drugB_name']]
        cell_feat = self.cell_features[sample['cell_line']]

        combo_feat = np.concatenate([drugA_feat, drugB_feat, cell_feat]).astype(np.float32)

        return {
            'drugA_feat': torch.from_numpy(drugA_feat),
            'drugB_feat': torch.from_numpy(drugB_feat),
            'cell_feat': torch.from_numpy(cell_feat),
            'combo_feat': torch.from_numpy(combo_feat),
            'target': torch.tensor(sample['target'], dtype=torch.float32),
            'drugA_conc': torch.tensor(sample['drugA_conc'], dtype=torch.float32),
            'drugB_conc': torch.tensor(sample['drugB_conc'], dtype=torch.float32),
            'single_resp_1': torch.tensor(sample['single_resp_1'], dtype=torch.float32),
            'single_resp_2': torch.tensor(sample['single_resp_2'], dtype=torch.float32),
            'index': idx,
            'drugA_name': sample['drugA_name'],
            'drugB_name': sample['drugB_name'],
            'cell_line': sample['cell_line']
        }

class DrugCombinationModel(nn.Module):
    def __init__(self, drug_feat_dim=1000, cell_feat_dim=977):
        super(DrugCombinationModel, self).__init__()

        self.drug_encoder = nn.Sequential(
            nn.Linear(drug_feat_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )

        self.cell_encoder = nn.Sequential(
            nn.Linear(cell_feat_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )

        self.theta1_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

        self.theta2_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

        self.epsilon_net = nn.Sequential(
            nn.Linear(256 * 3 + 256 + 2, 1024),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

        self.dose_encoder = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
        )

        hpb_input_dim = 256 + 256 + 256 + 32 + 32
        self.predictor_direct = nn.Sequential(
            nn.Linear(hpb_input_dim, 2048),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        drugA_feat,
        drugB_feat,
        cell_feat,
        combo_feat,
        drugA_conc,
        drugB_conc,
    ):
        encoded_drugA = self.drug_encoder(drugA_feat)
        encoded_drugB = self.drug_encoder(drugB_feat)
        encoded_cell = self.cell_encoder(cell_feat)

        theta1_input = torch.cat(
            [encoded_drugA, encoded_cell, drugA_conc.unsqueeze(1)], dim=1
        )
        theta2_input = torch.cat(
            [encoded_drugB, encoded_cell, drugB_conc.unsqueeze(1)], dim=1
        )
        theta1_raw = self.theta1_net(theta1_input)
        theta2_raw = self.theta2_net(theta2_input)

        drug_pair_sum = encoded_drugA + encoded_drugB
        drug_pair_product = encoded_drugA * encoded_drugB
        drug_pair_abs_difference = torch.abs(encoded_drugA - encoded_drugB)
        epsilon_input = torch.cat(
            [
                drug_pair_sum,
                drug_pair_product,
                drug_pair_abs_difference,
                encoded_cell,
                drugA_conc.unsqueeze(1),
                drugB_conc.unsqueeze(1),
            ],
            dim=1,
        )
        epsilon = self.epsilon_net(epsilon_input)

        theta12_sum = theta1_raw + theta2_raw
        theta1 = theta1_raw / theta12_sum
        theta2 = theta2_raw / theta12_sum

        encoded_doseA = self.dose_encoder(drugA_conc.unsqueeze(1))
        encoded_doseB = self.dose_encoder(drugB_conc.unsqueeze(1))

        combined_features = torch.cat(
            [
                encoded_drugA,
                encoded_drugB,
                encoded_cell,
                encoded_doseA,
                encoded_doseB,
            ],
            dim=1,
        )
        p_direct = self.predictor_direct(combined_features)

        return (
            theta1.squeeze(-1),
            theta2.squeeze(-1),
            epsilon.squeeze(-1),
            p_direct.squeeze(-1),
        )

def train_model(model, train_loader, val_loader, train_df, val_df, epochs=100, lr=5e-5,early_stop_patience=10,min_delta=1e-6):
    criterion = nn.MSELoss()
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
        min_lr=1e-7
    )

    best_val_loss = float('inf')
    no_improve_count = 0
    model = model.to(device)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            drugA_feat = batch['drugA_feat'].to(device)
            drugB_feat = batch['drugB_feat'].to(device)
            cell_feat = batch['cell_feat'].to(device)
            combo_feat = batch['combo_feat'].to(device)
            target = batch['target'].to(device)
            drugA_conc = batch['drugA_conc'].to(device)
            drugB_conc = batch['drugB_conc'].to(device)
            single_resp_1 = batch['single_resp_1'].to(device)
            single_resp_2 = batch['single_resp_2'].to(device)
            indices = batch['index'].numpy()

            optimizer.zero_grad()

            theta1, theta2, epsilon, P_direct = model(drugA_feat, drugB_feat, cell_feat, combo_feat, drugA_conc, drugB_conc)
            P_abg = theta1 * single_resp_1 + theta2 * single_resp_2 + epsilon
            P_final = 0.5 * P_abg + 0.5 * P_direct
            loss = criterion(P_final, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * drugA_feat.size(0)

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        val_loss, val_metrics, val_results = evaluate_model(model, val_loader, val_df, epoch + 1)
        val_losses.append(val_loss)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {current_lr:.8f}')
        print(f'Val Metrics - MSE: {val_metrics["mse"]:.4f}, RMSE: {val_metrics["rmse"]:.4f}, '
              f'MAE: {val_metrics["mae"]:.4f}, PCC: {val_metrics["pcc"]:.4f}, R2: {val_metrics["r2"]:.4f}')
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            no_improve_count = 0
            torch.save(model.state_dict(), 'best_model_cell_line_cold_start_seed42.pth')
            print(f"Saved best model at epoch {epoch+1}, Val Loss: {best_val_loss:.4f}")
        else:
            no_improve_count += 1
            print(f"No improvement count: {no_improve_count}/{early_stop_patience}")

        if no_improve_count >= early_stop_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    plt.figure(figsize=(10, 5))
    x_axis = range(1, len(train_losses) + 1)
    plt.plot(x_axis, train_losses, label='Train Loss')
    plt.plot(x_axis, val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss (1-X/X0)')
    plt.legend()
    plt.grid(True)
    plt.savefig('loss_curve_cell_line_cold_start_seed42.png')
    plt.close()

    model.load_state_dict(torch.load('best_model_cell_line_cold_start_seed42.pth', map_location=device))
    return model, train_losses, val_losses

def concordance_index(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    finite = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[finite], y_pred[finite]
    if len(y_true) < 2:
        return np.nan
    order = np.argsort(y_true, kind='mergesort')
    y_true, y_pred = y_true[order], y_pred[order]
    _, pred_rank = np.unique(y_pred, return_inverse=True)
    pred_rank = pred_rank + 1
    tree = np.zeros(pred_rank.max() + 1, dtype=np.int64)
    def bit_add(index):
        while index < len(tree):
            tree[index] += 1
            index += index & -index
    def bit_sum(index):
        total = 0
        while index > 0:
            total += tree[index]
            index -= index & -index
        return total
    concordant = 0.0
    comparable = 0
    previous_count = 0
    start = 0
    while start < len(y_true):
        end = start + 1
        while end < len(y_true) and y_true[end] == y_true[start]:
            end += 1
        for rank in pred_rank[start:end]:
            lower = bit_sum(rank - 1)
            equal = bit_sum(rank) - lower
            concordant += lower + 0.5 * equal
            comparable += previous_count
        for rank in pred_rank[start:end]:
            bit_add(rank)
        previous_count += end - start
        start = end
    return concordant / comparable if comparable else np.nan

def evaluate_model(model, data_loader, data_df, epoch, save_results=True):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    results = []

    with torch.no_grad():
        for batch in data_loader:
            drugA_feat = batch['drugA_feat'].to(device)
            drugB_feat = batch['drugB_feat'].to(device)
            cell_feat = batch['cell_feat'].to(device)
            combo_feat = batch['combo_feat'].to(device)
            target = batch['target'].to(device)
            drugA_conc = batch['drugA_conc'].to(device)
            drugB_conc = batch['drugB_conc'].to(device)
            single_resp_1 = batch['single_resp_1'].to(device)
            single_resp_2 = batch['single_resp_2'].to(device)
            indices = batch['index'].numpy()

            theta1, theta2, epsilon, P_direct = model(drugA_feat, drugB_feat, cell_feat, combo_feat, drugA_conc, drugB_conc)
            P_abg = theta1 * single_resp_1 + theta2 * single_resp_2 + epsilon
            P_final = 0.5 * P_abg + 0.5 * P_direct

            all_preds.extend(P_final.cpu().numpy().flatten())
            all_targets.extend(target.cpu().numpy().flatten())

            loss = nn.MSELoss()(P_final, target)
            total_loss += loss.item() * drugA_feat.size(0)

            theta1_np = theta1.cpu().numpy()
            theta2_np = theta2.cpu().numpy()
            epsilon_np = epsilon.cpu().numpy()
            pred_np = P_final.cpu().numpy()
            target_np = target.cpu().numpy()

            for j, idx in enumerate(indices):
                sample_data = data_df.iloc[idx]
                results.append({
                    'epoch': epoch,
                    'drugA_name': sample_data['drugA_name'],
                    'drugB_name': sample_data['drugB_name'],
                    'cell_line': sample_data['cell_line'],
                    'drugA_conc': sample_data['drugA_conc'],
                    'drugB_conc': sample_data['drugB_conc'],
                    'target': target_np[j],
                    'prediction': pred_np[j],
                    'single_resp_1': sample_data['single_resp_1'],
                    'single_resp_2': sample_data['single_resp_2'],
                    'theta1': theta1_np[j],
                    'theta2': theta2_np[j],
                    'epsilon': epsilon_np[j],
                    'set': 'validation'
                })

    total_loss /= len(data_loader.dataset)

    assert len(all_preds) == len(all_targets), f"预测值数量({len(all_preds)})与真实值数量({len(all_targets)})不匹配"

    mse = mean_squared_error(all_targets, all_preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(all_targets, all_preds)
    pcc, _ = pearsonr(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)

    metrics = {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'pcc': pcc,
        'r2': r2,
        'c_index': concordance_index(all_targets, all_preds)
    }

    return total_loss, metrics, results

def save_final_theta1_theta2_epsilon(model, data_loader, data_df):
    model.eval()
    results = []

    with torch.no_grad():
        for batch in data_loader:
            indices = batch['index'].numpy()
            batch_data = data_df.iloc[indices]

            drugA_feat = batch['drugA_feat'].to(device)
            drugB_feat = batch['drugB_feat'].to(device)
            cell_feat = batch['cell_feat'].to(device)
            combo_feat = batch['combo_feat'].to(device)
            drugA_conc = batch['drugA_conc'].to(device)
            drugB_conc = batch['drugB_conc'].to(device)
            target = batch['target'].to(device)
            single_resp_1 = batch['single_resp_1'].to(device)
            single_resp_2 = batch['single_resp_2'].to(device)

            theta1, theta2, epsilon, P_direct = model(drugA_feat, drugB_feat, cell_feat, combo_feat, drugA_conc, drugB_conc)
            P_abg = theta1 * single_resp_1 + theta2 * single_resp_2 + epsilon
            P_final = 0.5 * P_abg + 0.5 * P_direct

            theta1_np = theta1.cpu().numpy().flatten()
            theta2_np = theta2.cpu().numpy().flatten()
            epsilon_np = epsilon.cpu().numpy().flatten()
            pred_np = P_final.cpu().numpy().flatten()
            target_np = target.cpu().numpy().flatten()

            for j in range(len(theta1_np)):
                sample_data = batch_data.iloc[j]
                results.append({
                    'drugA_name': sample_data['drugA_name'],
                    'drugB_name': sample_data['drugB_name'],
                    'cell_line': sample_data['cell_line'],
                    'drugA_conc': sample_data['drugA_conc'],
                    'drugB_conc': sample_data['drugB_conc'],
                    'target': target_np[j],
                    'prediction': pred_np[j],
                    'single_resp_1': sample_data['single_resp_1'],
                    'single_resp_2': sample_data['single_resp_2'],
                    'theta1': theta1_np[j],
                    'theta2': theta2_np[j],
                    'epsilon': epsilon_np[j]
                })

    results_df = pd.DataFrame(results)
    column_order = [
        'drugA_name', 'drugB_name', 'cell_line',
        'drugA_conc', 'drugB_conc',
        'target', 'prediction',
        'single_resp_1', 'single_resp_2',
        'theta1', 'theta2', 'epsilon'
    ]
    results_df = results_df[column_order]
    results_df.to_csv('final_cell_line_cold_start_test_predictions_seed42.csv', index=False)
    print("Saved final theta1, theta2, epsilon values with drug and cell line info (1-X/X0)")

def main():
    combo_df, drug_features, cell_features = load_data()
    data_df = preprocess_data(combo_df, drug_features, cell_features)

    if len(data_df) == 0:
        print("No valid data samples after preprocessing. Exiting.")
        return

    sample = data_df.iloc[0]
    drug_feat_dim = drug_features.iloc[0].values[2:].astype(np.float32).shape[0]
    cell_feat_dim = cell_features.iloc[0].values.astype(np.float32).shape[0]
    print(f"Drug feature dimension: {drug_feat_dim}")
    print(f"Cell feature dimension: {cell_feat_dim}")

    train_df, val_df, test_df, split_summary = cell_line_train_val_test_split(
        data_df, train_size=0.70, val_size=0.10, test_size=0.20, random_state=42
    )

    train_dataset = DrugCombinationDataset(train_df, drug_features, cell_features)
    val_dataset = DrugCombinationDataset(val_df, drug_features, cell_features)
    test_dataset = DrugCombinationDataset(test_df, drug_features, cell_features)
    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset)}")
    print(f"Test dataset size: {len(test_dataset)}")

    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED)
    train_loader = DataLoader(
        train_dataset, batch_size=8, shuffle=True, generator=loader_generator
    )
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

    model = DrugCombinationModel(drug_feat_dim=drug_feat_dim, cell_feat_dim=cell_feat_dim)

    trained_model, train_losses, val_losses = train_model(
        model, train_loader, val_loader, train_df, val_df, epochs=100, lr=5e-5
    )

    final_loss, final_metrics, _ = evaluate_model(trained_model, test_loader, test_df, 100)
    print(f'\nFinal Test Metrics (1-X/X0):')
    print(f'MSE: {final_metrics["mse"]:.4f}')
    print(f'RMSE: {final_metrics["rmse"]:.4f}')
    print(f'MAE: {final_metrics["mae"]:.4f}')
    print(f'PCC: {final_metrics["pcc"]:.4f}')
    print(f'R2: {final_metrics["r2"]:.4f}')
    print(f'C-index: {final_metrics["c_index"]:.4f}')
    metric_record = {
        'seed': SEED,
        **final_metrics,
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
        'train_cell_lines': train_df['cell_line'].nunique(),
        'val_cell_lines': val_df['cell_line'].nunique(),
        'test_cell_lines': test_df['cell_line'].nunique()
    }
    pd.DataFrame([metric_record]).to_csv(
        f'cell_line_cold_start_metrics_seed{SEED}.csv', index=False, encoding='utf-8-sig'
    )

    torch.save(trained_model.state_dict(), 'final_model_cell_line_cold_start_seed42.pth')
    save_final_theta1_theta2_epsilon(trained_model, test_loader, test_df)

if __name__ == '__main__':
    main()
